# Experiment plots exporter

Loads result pickles (same format as `powerbi_exporter.ipynb`) and writes comparison plots as **PNG** and **PDF** under `notebooks/result_plots/`.

Typical use: set `ARCHIVES` to the archive id(s) you ran (e.g. `[108]`), run all cells, then open the output folder.

In [ ]:
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

In [ ]:
# ===================== CONFIG =====================
ARCHIVES = [108]
RESULT_ROOT = Path("../result")
OUTPUT_DIR = Path("./result_plots")
SAVE_FORMATS = ("png", "pdf")  # both written for each figure
INCLUDE_TLE = False
DPI = 200

# Metrics to plot (column name -> y-axis label)
METRICS = {
    "Objective_f(S)": "Objective f(S)",
    "Node_Count": "Nodes popped",
    "Time_s": "Wall time (s)",
}

# Short names for legends
ALGO_LABELS = {
    "EfficientBFS": "EfficientBFS",
    "AdaptiveEfficientBFS": "AdaptiveBFS",
    "EfficientBranchAndBound": "EBB",
    "BFSTC": "BFSTC",
}

UB_LINESTYLES = {"ub0": "-", "ub2": "--"}
UB_MARKERS = {"ub0": "o", "ub2": "s"}
# =================================================

In [ ]:
def discover_pickle_files(archives, result_root: Path):
    files = []
    for a in archives:
        root = result_root / f"archive-{a}"
        if not root.exists():
            print(f"[WARN] Missing archive path: {root}")
            continue
        files.extend(root.rglob("*.pckl"))
    return sorted(files)


def parse_legacy_from_path_and_name(p: Path):
    task = ground_size = seed = None
    try:
        seed = int(p.parent.name)
        ground_size = int(p.parent.parent.name)
        task = p.parent.parent.parent.name
    except Exception:
        pass

    stem = p.stem
    parts = stem.split("-")
    algo = strategy = ub = d_mode = budget = alpha = model = None

    KNOWN_ALGOS_7PART = (
        "EfficientBFS",
        "AdaptiveEfficientBFS",
        "BFSTC",
        "EfficientBranchAndBound",
    )

    if len(parts) >= 6:
        model = parts[-1]
        alpha = float(parts[-2]) if parts[-2].replace(".", "", 1).isdigit() else None
        budget = float(parts[-3]) if parts[-3].replace(".", "", 1).isdigit() else None
        if parts[0] in KNOWN_ALGOS_7PART and len(parts) >= 7:
            algo = parts[0]
            strategy = parts[1] if parts[1] != "none" else None
            ub = parts[2]
            d_mode = parts[3]
        elif parts[0] in ("BFSTC", "Efficient") and len(parts) >= 6:
            algo = parts[0]
            strategy = None
            ub = parts[1]
            d_mode = parts[2]

    return {
        "Task": task,
        "Ground_Size": ground_size,
        "Seed": seed,
        "Algorithm": algo,
        "Strategy": strategy,
        "UB": ub,
        "D": d_mode,
        "Budget": budget,
        "Alpha": alpha,
        "Model": model,
    }


def build_row_from_pickle(p: Path):
    with p.open("rb") as f:
        res = pickle.load(f)

    meta = res.get("meta") if isinstance(res, dict) else None
    legacy = parse_legacy_from_path_and_name(p)

    row = {
        "Source_File": str(p),
        "Archive": None,
        "Task": legacy.get("Task"),
        "Ground_Size": legacy.get("Ground_Size"),
        "Seed": legacy.get("Seed"),
        "Algorithm": legacy.get("Algorithm"),
        "Strategy": legacy.get("Strategy"),
        "UB": legacy.get("UB"),
        "D": legacy.get("D"),
        "Budget": legacy.get("Budget"),
        "Alpha": legacy.get("Alpha"),
        "Model": legacy.get("Model"),
        "Objective_f(S)": res.get("f(S)") if isinstance(res, dict) else None,
        "Cost_c(S)": res.get("c(S)") if isinstance(res, dict) else None,
        "Time_s": res.get("time") if isinstance(res, dict) else None,
        "Node_Count": res.get("node_count") if isinstance(res, dict) else None,
        "Open_List_Count": res.get("open_list_count") if isinstance(res, dict) else None,
        "TLE": res.get("TLE") if isinstance(res, dict) else None,
        "Solution_Set_Size": len(res.get("S", [])) if isinstance(res, dict) else None,
    }

    if isinstance(meta, dict):
        row["Archive"] = meta.get("archive")
        row["Task"] = meta.get("task", row["Task"])
        row["Ground_Size"] = meta.get("ground_size", row["Ground_Size"])
        row["Seed"] = meta.get("seed", row["Seed"])
        row["Algorithm"] = meta.get("algorithm", row["Algorithm"])
        row["Strategy"] = meta.get("strategy", row["Strategy"])
        row["UB"] = meta.get("heuristic", row["UB"])
        row["D"] = meta.get("d", row["D"])
        row["Budget"] = meta.get("budget", row["Budget"])
        row["Alpha"] = meta.get("alpha", row["Alpha"])
        row["Model"] = meta.get("model_class", row["Model"])
    else:
        m = re.search(r"archive-(\d+)", str(p).replace("\\", "/"))
        if m:
            row["Archive"] = m.group(1)

    return row


def load_experiment_dataframe(archives, result_root: Path, include_tle=True):
    files = discover_pickle_files(archives, result_root)
    print(f"[INFO] Found {len(files)} pickle files across archives={archives}")

    rows = []
    errors = []
    for p in tqdm(files, desc="Parsing pickles"):
        try:
            row = build_row_from_pickle(p)
            if (not include_tle) and bool(row.get("TLE", False)):
                continue
            rows.append(row)
        except Exception as e:
            errors.append({"file": str(p), "error": str(e)})

    df = pd.DataFrame(rows)
    if df.empty:
        return df, errors

    num_cols = [
        "Archive", "Ground_Size", "Seed", "Budget", "Alpha",
        "Objective_f(S)", "Cost_c(S)", "Time_s", "Node_Count", "Open_List_Count",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["algo_label"] = df["Algorithm"].map(lambda a: ALGO_LABELS.get(a, a))
    df["series"] = df.apply(
        lambda r: f"{r['algo_label']} ({r['UB']})",
        axis=1,
    )
    df.sort_values(
        by=["Archive", "Task", "Algorithm", "UB", "Alpha", "Budget", "Seed"],
        inplace=True,
        na_position="last",
    )
    return df, errors

In [ ]:
def _save_figure(fig, out_base: Path):
    out_base.parent.mkdir(parents=True, exist_ok=True)
    for fmt in SAVE_FORMATS:
        path = out_base.with_suffix(f".{fmt}")
        kw = {"bbox_inches": "tight"}
        if fmt == "png":
            kw["dpi"] = DPI
        fig.savefig(path, **kw)
    plt.close(fig)


def plot_metric_vs_budget(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    out_base: Path,
    *,
    hue: str = "series",
    title=None,
):
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, sub in df.groupby(hue, sort=False):
        sub = sub.sort_values("Budget")
        ub = sub["UB"].iloc[0]
        ax.plot(
            sub["Budget"],
            sub[metric],
            label=name,
            linestyle=UB_LINESTYLES.get(ub, "-"),
            marker=UB_MARKERS.get(ub, "o"),
            linewidth=2,
        )
    ax.set_xlabel("Budget")
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(title="Algorithm (UB)", fontsize=9, loc="best")
    _save_figure(fig, out_base)


def plot_metric_by_task_alpha(
    df: pd.DataFrame,
    metric: str,
    out_dir: Path,
):
    for task in sorted(df["Task"].dropna().unique()):
        for alpha in sorted(df["Alpha"].dropna().unique()):
            sub = df[(df["Task"] == task) & (df["Alpha"] == alpha)]
            if sub.empty:
                continue
            out_base = out_dir / f"{task}_{metric}_alpha{alpha:g}"
            plot_metric_vs_budget(
                sub,
                metric,
                METRICS[metric],
                out_base,
                hue="series",
                title=f"{task} | α={alpha:g} | {METRICS[metric]}",
            )

In [ ]:
def plot_nodes_log_scale(
    df: pd.DataFrame,
    out_dir: Path,
    min_nodes: float = 1.0,
):
    sub = df[df["Node_Count"] >= min_nodes].copy()
    if sub.empty:
        print("[WARN] No rows with Node_Count >= 1 for log-scale plot")
        return
    for task in sorted(sub["Task"].dropna().unique()):
        for alpha in sorted(sub["Alpha"].dropna().unique()):
            part = sub[(sub["Task"] == task) & (sub["Alpha"] == alpha)]
            if part.empty:
                continue
            fig, ax = plt.subplots(figsize=(8, 5))
            for name, grp in part.groupby("series", sort=False):
                grp = grp.sort_values("Budget")
                ub = grp["UB"].iloc[0]
                ax.plot(
                    grp["Budget"],
                    grp["Node_Count"],
                    label=name,
                    linestyle=UB_LINESTYLES.get(ub, "-"),
                    marker=UB_MARKERS.get(ub, "o"),
                    linewidth=2,
                )
            ax.set_yscale("log")
            ax.set_xlabel("Budget")
            ax.set_ylabel("Nodes popped (log scale)")
            ax.set_title(f"{task} | α={alpha:g} | nodes (log)")
            ax.legend(fontsize=9, loc="best")
            _save_figure(fig, out_dir / f"{task}_nodes_log_alpha{alpha:g}")

In [ ]:
def plot_bars_at_budget(
    df: pd.DataFrame,
    budget: float,
    metric: str,
    out_base: Path,
    title=None,
):
    sub = df[df["Budget"] == budget].copy()
    if sub.empty:
        return
    algos = sorted(sub["Algorithm"].dropna().unique())
    x_labels = [ALGO_LABELS.get(a, a) for a in algos]
    x = range(len(algos))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(algos) * width + 1), 5))
    for i, ub in enumerate(sorted(sub["UB"].dropna().unique())):
        means = []
        for algo in algos:
            m = sub[(sub["Algorithm"] == algo) & (sub["UB"] == ub)][metric].mean()
            means.append(float(m) if pd.notna(m) else 0.0)
        off = (i - 0.5 * (len(sub["UB"].dropna().unique()) - 1)) * width
        ax.bar([xi + off for xi in x], means, width=width, label=str(ub))
    ax.set_xticks(list(x), x_labels)
    ax.set_ylabel(METRICS[metric])
    if title:
        ax.set_title(title)
    ax.legend(title="UB")
    _save_figure(fig, out_base)

In [ ]:
df_raw, parse_errors = load_experiment_dataframe(ARCHIVES, RESULT_ROOT, include_tle=INCLUDE_TLE)
print(f"[OK] Loaded {len(df_raw)} rows")

# Average over seeds for line/bar plots
agg_cols = ["Objective_f(S)", "Node_Count", "Time_s", "Cost_c(S)"]
df = (
    df_raw.groupby(
        ["Archive", "Task", "Ground_Size", "Algorithm", "Strategy", "UB", "Alpha", "Budget"],
        as_index=False,
    )[agg_cols]
    .mean(numeric_only=True)
)
df["algo_label"] = df["Algorithm"].map(lambda a: ALGO_LABELS.get(a, a))
df["series"] = df.apply(lambda r: f"{r['algo_label']} ({r['UB']})", axis=1)
if parse_errors:
    err_path = OUTPUT_DIR / f"archive-{'-'.join(str(a) for a in ARCHIVES)}_parse_errors.csv"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(parse_errors).to_csv(err_path, index=False)
    print(f"[WARN] {len(parse_errors)} parse errors -> {err_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(OUTPUT_DIR / "experiment_summary_raw.csv", index=False)
df.to_csv(OUTPUT_DIR / "experiment_summary.csv", index=False)
print(df.groupby(["Task", "Algorithm", "UB", "Alpha"]).size().head(20))

In [ ]:
out_dir = OUTPUT_DIR / f"archive-{'-'.join(str(a) for a in ARCHIVES)}"
out_dir.mkdir(parents=True, exist_ok=True)

for metric, ylabel in METRICS.items():
    plot_metric_by_task_alpha(df, metric, out_dir)

plot_nodes_log_scale(df, out_dir)

for b in sorted(df["Budget"].dropna().unique()):
    for metric, ylabel in METRICS.items():
        plot_bars_at_budget(
            df,
            float(b),
            metric,
            out_dir / f"bars_budget{b:g}_{metric}",
            title=f"Budget {b:g} | {ylabel}",
        )

print(f"[DONE] Figures written to {out_dir.resolve()}")